In [ ]:
import os, time, math as dt
from datetime import datetime
import dotenv
import geopandas as gpd
import pandas as pd
import numpy as np
import logging
import gc 
from glob import glob
import xlsxwriter
import openpyxl
import matplotlib.pyplot as plt
from io import BytesIO
import re
import matplotlib
import shapely
from shapely import force_2d
from shapely.prepared import prep
from shapely.geometry import LineString, MultiLineString
from shapely.ops import polygonize
from shapely.geometry import box
from shapely.validation import make_valid
import rapidfuzz


gc.enable()

dotenv.load_dotenv()


wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
output_parquets=os.getenv("OUTPUT_PARQUETS")
marxan_inputs=os.getenv("MARXAN_INPUTS")
input_parquets=os.getenv("INPUT_PARQUETS")
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
mineral_licks_gdb=os.path.join(marxan_inputs,'MineralLicks_Hex_2024_12_16','MineralLicks_Hex_2024_12_16.gdb')
licks_layer='MineralLicks_2024_12_16'
grid_path=os.path.join(input_parquets,'hexagons_3ha.parquet')
grid_id_col='GRID_ID'
cia1_path=os.path.join(input_parquets,'cia_1.parquet')
cia2_path=os.path.join(input_parquets,'cia_2.parquet')
weighted_surface_path = os.path.join(output_dir, "step_3f_weighted_surface.parquet")
scenario5_setup_gdb=os.path.join(source_data,'Scenario5_Setup.gdb')
wmb_layer='priority_WMB'
rec_for_path=os.path.join(output_dir,'step_3c_Recruitment_Forest.parquet')
aflb_parquet=os.path.join(input_parquets,'thlb_select.parquet')
step2_lyr=os.path.join(output_parquets,'ret_class_2e_all.parquet')
target_crs = "EPSG:3005"
wmv_inputs=os.getenv("WMV_INPUTS")
ret_class=os.path.join(input_parquets,'rec_class_all.parquet')
table_outputs=os.getenv('TABLE_OUPUTS')
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

dt_today=datetime.today().strftime('%Y%m%d')
out_path = os.path.join(table_outputs,f"scenario_5_summary_{dt_today}.xlsx")

scen5_run_number='5a'
scen5_run_folder='brwmb_parquets'
in_pars=os.path.join(wrk_dir,scen5_run_folder)

input_data_gdb=os.path.join(source_data,'scen_5_inputs.gdb')
consrv_lnd_path=os.path.join(in_pars,'step_3o_conservation_lands.parquet')  
scndry_rsrv_path=os.path.join(in_pars,'step_3i_secondary_reserves.parquet')
aspa_def_path=os.path.join(in_pars,'step_3j_aspatial_deferred_old_forest.parquet')
step_2e_path=os.path.join(output_dir, "ret_class_2e_all.parquet")

hv1_l='hv1_clip'

eco_png=gdb=os.path.join(source_data,'ECONOMIC_PNG_Critical_Infrastructure_3March2026.gdb')
eco_png_layer=''

#base tables
table_path=os.path.join(source_data,'spreadsheets')
wmb_table=os.path.join(table_path,'wmb_areas.xlsx')
tsu_table=os.path.join(table_path,'tsu_areas.xlsx')
recuritment_table=os.path.join(table_path,'wmb_recuirtment.xlsx')
wmb_for_rep_path=os.path.join(table_path,'wmb_for_rep.xlsx')

#layers for summary tables
rec_2e_all_path=os.path.join(output_dir,'ret_class_2e_all.parquet') 
scnd_rsrv_path=os.path.join(output_dir,'step_3i_secondary_reserves.parquet')
aspa_def_path=os.path.join(output_dir,'step_3j_aspatial_deferred_old_forest.parquet')
consrv_lnd_path=os.path.join(output_parquets,'step_3o_conservation_lands.parquet')


In [ ]:
#load in tables 
wmb_df=pd.read_excel(wmb_table)
tsu_df=pd.read_excel(tsu_table)
recruitment_df=pd.read_excel(recuritment_table)
wmb_for_rep_df=pd.read_excel(wmb_for_rep_path)

In [ ]:
#load in layers for 2e, secondary reserves, aspatial deferred old forest, conservation lands
rec_2e_all_df=gpd.read_parquet(rec_2e_all_path)
scnd_rsrv_df=gpd.read_parquet(scndry_rsrv_path)
aspa_def_df=gpd.read_parquet(aspa_def_path)
consrv_lnd_df=gpd.read_parquet(consrv_lnd_path)

In [ ]:

#WATER_MANAGEMENT_BASIN_NAME
rec_2e_all_df = rec_2e_all_df.drop(columns=['aflb_fact'])
if 'aflb_fact_bfrn' in rec_2e_all_df.columns:
    rec_2e_all_df.rename(columns={'aflb_fact_bfrn':'aflb_fact'}, inplace=True)  
rec_2e_all_df.loc[rec_2e_all_df['Rec_Cat'] > 0, 'aflb_fact'] = 1
rec_2e_all_df['area_ha']=rec_2e_all_df.geometry.area/10000
rec_2e_all_df['thlb_area_ha']=rec_2e_all_df['thlb_fact']*rec_2e_all_df['area_ha']
rec_2e_all_df['aflb_ha']=rec_2e_all_df['aflb_fact']*rec_2e_all_df['area_ha']
rec_2e_all_df.rename(columns={'WATER_MANAGEMENT_BASIN_NAME':'NAME'}, inplace=True)

if 'aflb_fact_bfrn' in scnd_rsrv_df.columns:
    scnd_rsrv_df.rename(columns={'aflb_fact_bfrn':'aflb_fact'}, inplace=True)  
    
scnd_rsrv_df['area_ha']=scnd_rsrv_df.geometry.area/10000
scnd_rsrv_df['thlb_area_ha']=scnd_rsrv_df['thlb_fact']*scnd_rsrv_df['area_ha']
scnd_rsrv_df['aflb_ha']=scnd_rsrv_df['aflb_fact']*scnd_rsrv_df['area_ha']

if 'aflb_fact_bfrn' in aspa_def_df.columns:
    aspa_def_df.rename(columns={'aflb_fact_bfrn':'aflb_fact'}, inplace=True)  
aspa_def_df['area_ha']=aspa_def_df.geometry.area/10000
aspa_def_df['thlb_area_ha']=aspa_def_df['thlb_fact']*aspa_def_df['area_ha']
aspa_def_df['aflb_ha']=aspa_def_df['aflb_fact']*aspa_def_df['area_ha']

if 'aflb_fact_bfrn' in aspa_def_df.columns:
    aspa_def_df.rename(columns={'aflb_fact_bfrn':'aflb_fact'}, inplace=True)  
consrv_lnd_df['area_ha']=consrv_lnd_df.geometry.area/10000
consrv_lnd_df['thlb_area_ha']=consrv_lnd_df['thlb_fact']*consrv_lnd_df['area_ha']
consrv_lnd_df['aflb_ha']=consrv_lnd_df['aflb_fact']*consrv_lnd_df['area_ha']
   

#### 3.	Does the merged layer from 2 e) amount to 25% of each WMB AFLB and 33% of each TSU AFLB? (TSUs are the clipped portion of each trapline that occur inside a given WMB)

In [ ]:
wmb_rec_2e = (
    rec_2e_all_df
    .groupby(['NAME'], as_index=False)
    .agg(
        merged_layer_2e_area_ha=('area_ha', 'sum'),
        merged_layer_2e_AFLB_ha=('aflb_ha', 'sum')
    )
)
accounting_3_wmb = wmb_df.merge(
    wmb_rec_2e,
    on=['NAME'],
    how='left'
)
accounting_3_wmb['difference_ha']=accounting_3_wmb['aflb_25_percent']-accounting_3_wmb['merged_layer_2e_AFLB_ha']
accounting_3_wmb['coverage_pct_of_aflb']=accounting_3_wmb['merged_layer_2e_AFLB_ha']/accounting_3_wmb['SUM_aflb_ha']*100  
accounting_3_wmb['meets_25_percent_aflb']=accounting_3_wmb['coverage_pct_of_aflb']>=25  

In [ ]:
#need one for tsu. joining outside of and bring back in 

wmb_rec_2e_tsu = (
    rec_2e_all_df
    .groupby(['NAME','TRAPLINE_AREA_IDENTIFIER'], as_index=False)
    .agg(
        merged_layer_2e_area_ha=('area_ha', 'sum'),
        merged_layer_2e_AFLB_ha=('aflb_ha', 'sum')
    )
)
accounting_3_tsu = tsu_df.merge(
    wmb_rec_2e_tsu,
    on=['NAME', 'TRAPLINE_AREA_IDENTIFIER'],
    how='left'
)

accounting_3_tsu['difference_ha']=accounting_3_tsu['tsu_33_aaflb_ha']-accounting_3_tsu['merged_layer_2e_AFLB_ha']
accounting_3_tsu['coverage_pct_of_aflb']=accounting_3_tsu['merged_layer_2e_AFLB_ha']/accounting_3_tsu['tsu_aflb_ha']*100  
accounting_3_tsu['meets_25_percent_aflb']=accounting_3_tsu['coverage_pct_of_aflb']>=33  

#### 4.	Does the merged layer in 2 e) achieve the desired ‘total area’ for each Rec_Cat/FOR REP described at the bottom of the tables provided for the 4 WMBs? 

In [ ]:
wmb_rec_2e = (
    rec_2e_all_df
    .groupby(['NAME','Rec_Cat_short'], as_index=False)
    .agg(
        merged_layer_2e_AFLB_ha=('aflb_ha', 'sum')
    )
)


wmb_rec_2e['NAME'] = "2e " + wmb_rec_2e['NAME']

wmb_rec_2e_wide=wmb_rec_2e.pivot(index='NAME', columns='Rec_Cat_short', values='merged_layer_2e_AFLB_ha').reset_index()

wmb_rec_2e_wide['NAME_clean'] = (
    wmb_rec_2e_wide['NAME']
    .str.replace(r"^2e\s+", "", regex=True)
    .str.upper()
)
recruitment_df['NAME_clean'] = recruitment_df['Unnamed: 0'].str.upper()


def clean_name(x):
    return (
        x.replace(" RIVER", "")
         .strip()
         .upper()
    )

wmb_rec_2e_wide["NAME_clean"] = wmb_rec_2e_wide["NAME_clean"].apply(clean_name)
recruitment_df["NAME_clean"] = recruitment_df["NAME_clean"].apply(clean_name)


cols = ["HPC","HPD","HPM","MPC","MPD","MPM","LPC","LPD","LPM"]
wmb_rec_2e_wide = wmb_rec_2e_wide[["NAME","NAME_clean"] + cols]
recruitment_df = recruitment_df[["Unnamed: 0","NAME_clean"] + cols + ["Total"]]

merged = recruitment_df.merge(
    wmb_rec_2e_wide,
    on="NAME_clean",
    how="left",
    suffixes=("_orig", "_2e")
)
for c in cols:
    merged[f"{c}_diff"] = merged[f"{c}_orig"] - merged[f"{c}_2e"]

merged["Total_2e"] = merged[[f"{c}_2e" for c in cols]].sum(axis=1)
    
merged["Total_diff"] = merged["Total"] - merged["Total_2e"]

final_rows = []


for _, row in merged.iterrows():
    base_name = row["Unnamed: 0"]

    # --- Original row ---
    orig_row = [base_name] + [row[f"{c}_orig"] for c in cols] + [row["Total"]]
    final_rows.append(orig_row)

    # --- 2e row ---
    name_2e = f"2e {base_name}"
    total_2e = row[[f"{c}_2e" for c in cols]].sum()
    row_2e = [name_2e] + [row[f"{c}_2e"] for c in cols] + [total_2e]
    final_rows.append(row_2e)

    # --- diff row ---
    diff_row = ["diff"] + [row[f"{c}_diff"] for c in cols] + [row["Total_diff"]]
    final_rows.append(diff_row)
    
    accounting_4_wmb_recruitment = pd.DataFrame(final_rows, columns=["NAME"] + cols + ["Total"])

#### 5.	Do the Secondary Reserves and Aspatially Deferred Old Forest amount to 25% of each WMB AFLB and 33% of each TSU AFLB? step3k outputs

In [ ]:
if 'NAME_1' in scnd_rsrv_df.columns:
    scnd_rsrv_df.rename(columns={'NAME_1':'NAME'}, inplace=True)    
if 'NAME_12' in scnd_rsrv_df.columns:
    scnd_rsrv_df.rename(columns={'NAME_12':'NAME'}, inplace=True)
if 'NAME_1' in aspa_def_df.columns:
    aspa_def_df.rename(columns={'NAME_1':'NAME'}, inplace=True)
if 'NAME_1' in consrv_lnd_df.columns:
    consrv_lnd_df.rename(columns={'NAME_1':'NAME'}, inplace=True)
    
wmb_scnd_rsrv_df = (
    scnd_rsrv_df
    .groupby(['NAME'], as_index=False)
    .agg(
        secondary_reserve_area_ha=('aflb_ha', 'sum'),
        secondary_reserve_aflb_ha=('aflb_ha', 'sum')
    )
)

wmb_aspa_def_df = (
    aspa_def_df
    .groupby(['NAME'], as_index=False)
    .agg(
        aspa_def_area_ha=('aflb_ha', 'sum'),
        aspa_def_aflb_ha=('aflb_ha', 'sum')
    )
)


accounting_5_wmb = (
    wmb_df
    .merge(wmb_scnd_rsrv_df, on='NAME', how='left')
    .merge(wmb_aspa_def_df, on='NAME', how='left')
)

accounting_5_wmb['selected_total_ha']=accounting_5_wmb['aspa_def_aflb_ha']+accounting_5_wmb['secondary_reserve_aflb_ha']
accounting_5_wmb['difference_ha']=accounting_5_wmb['aflb_25_percent']-accounting_5_wmb['selected_total_ha']
accounting_5_wmb['coverage_pct_of_aflb']=accounting_5_wmb['selected_total_ha']/accounting_5_wmb['SUM_aflb_ha']*100  
accounting_5_wmb['meets_25_percent_aflb']=accounting_5_wmb['coverage_pct_of_aflb']>=25  


wmb_scnd_rsrv_df = (
    scnd_rsrv_df
    .groupby(['NAME', 'TRAPLINE_AREA_IDENTIFIER'], as_index=False)
    .agg(
        secondary_reserve_area_ha=('area_ha', 'sum'),
        secondary_reserve_aflb_ha=('aflb_ha', 'sum')
    )
)

wmb_aspa_def_df = (
    aspa_def_df
    .groupby(['NAME', 'TRAPLINE_AREA_IDENTIFIER'], as_index=False)
    .agg(
        aspa_def_area_ha=('area_ha', 'sum'),
        aspa_def_aflb_ha=('aflb_ha', 'sum')
    )
)

accounting_5_tsu = (
    tsu_df
    .merge(wmb_scnd_rsrv_df, on=['NAME', 'TRAPLINE_AREA_IDENTIFIER'], how='left')
    .merge(wmb_aspa_def_df, on=['NAME', 'TRAPLINE_AREA_IDENTIFIER'], how='left')
)

accounting_5_tsu['selected_total_ha']=accounting_5_tsu['aspa_def_aflb_ha']+accounting_5_tsu['secondary_reserve_aflb_ha']
accounting_5_tsu['difference_ha']=accounting_5_tsu['tsu_33_aaflb_ha']-accounting_5_tsu['selected_total_ha']
accounting_5_tsu['coverage_pct_of_aflb']=accounting_5_tsu['selected_total_ha']/accounting_5_tsu['tsu_aflb_ha']*100  
accounting_5_tsu['meets_25_percent_aflb']=accounting_5_tsu['coverage_pct_of_aflb']>=25  





#### 6.	What is the total (all WMBs summed) THLB impact of the portion of Secondary Reserves and Aspatially Deferred Old Forest that occur outside HV1 boundaires?

In [ ]:
# read hv1 
hv1 = gpd.read_file(input_data_gdb, layer=hv1_l)

hv1_sindex = hv1.sindex

aspa_def_joined = aspa_def_df.sjoin(hv1, how="left", predicate="intersects")
aspa_missing = aspa_def_joined[aspa_def_joined["index_right"].isna()]

scnd_rsrv_joined = scnd_rsrv_df.sjoin(hv1, how="left", predicate="intersects")
scnd_rsrv_missing = scnd_rsrv_joined[scnd_rsrv_joined["index_right"].isna()]

rec_2e_joined= rec_2e_all_df.sjoin(hv1, how="left", predicate="intersects")
rec_2e_missing = rec_2e_joined[rec_2e_joined["index_right"].isna()]

scnd_rsrv_thlb_outside_hv1=scnd_rsrv_missing['thlb_area_ha'].sum()
scnd_rsrv_thlb=scnd_rsrv_df['thlb_area_ha'].sum()

scnd_rsrv_area_outside_hv1=scnd_rsrv_missing['area_ha_left'].sum()
scnd_rsrv_area=scnd_rsrv_df['area_ha'].sum()

aspa_def_thlb_outside_hv1=aspa_missing['thlb_area_ha'].sum()
aspa_def_thlb=aspa_def_df['thlb_area_ha'].sum()
aspa_def_area_outside_hv1=aspa_missing['area_ha_left'].sum()
aspa_def_area=aspa_def_df['area_ha'].sum()

rec_2e_thlb_outside_hv1=rec_2e_missing['thlb_area_ha'].sum()
rec_2e_thlb=rec_2e_all_df['thlb_area_ha'].sum()
rec_2e_area_outside_hv1=rec_2e_missing['area_ha_left'].sum()
rec_2e_area=rec_2e_all_df['area_ha'].sum()

#dictionary for new summary table
summary_dict = {
    "scnd_rsrv_area_outside_hv1": scnd_rsrv_area_outside_hv1,
    "aspa_def_area_outside_hv1": aspa_def_area_outside_hv1,
    "combined_area_outside_hv1": scnd_rsrv_area_outside_hv1 + aspa_def_area_outside_hv1,

    "scnd_rsrv_thlb_outside_hv1": scnd_rsrv_thlb_outside_hv1,
    "aspa_def_thlb_outside_hv1": aspa_def_thlb_outside_hv1,
    "combined_thlb_outside_hv1": scnd_rsrv_thlb_outside_hv1 + aspa_def_thlb_outside_hv1,
    
    "scnd_rsrv_area": scnd_rsrv_area,
    "aspa_def_area": aspa_def_area,
    "combined_area": scnd_rsrv_area + aspa_def_area,
    
    "scnd_rsrv_thlb": scnd_rsrv_thlb,
    "aspa_def_thlb": aspa_def_thlb,
    "combined_thlb_total": scnd_rsrv_thlb + aspa_def_thlb,
    
    "rec_2e_area_outside_hv1": rec_2e_area_outside_hv1,
    "rec_2e_area": rec_2e_area,
    "rec_2e_thlb_outside_hv1": rec_2e_thlb_outside_hv1,
    "rec_2e_thlb": rec_2e_thlb
}   

account_6_thlb=pd.DataFrame([summary_dict])

#### 7.	How much of each Rec_Cat/FOR REP category is represented by Secondary Reserves and Aspatially Deferred Old Forest for each WMB?

In [ ]:
scnd_rsrv_for_rep=scnd_rsrv_df.groupby(['NAME','Rec_Cat','Rec_Cat_short'], as_index=False).agg(
    secondary_reserve_aflb_ha=('aflb_ha', 'sum')
)

aspa_def_df_for_rep=aspa_def_df.groupby(['NAME','Rec_Cat','Rec_Cat_short'], as_index=False).agg(
    aspatially_deferred_aflb_ha=('aflb_ha', 'sum')
)

if 'NAME' not in wmb_for_rep_df.columns:
    wmb_for_rep_df.rename(columns={'WATER_MANAGEMENT_BASIN_NAME':'NAME'}, inplace=True)

accoutning_7_for_rep_rec_cat=wmb_for_rep_df.merge(
    scnd_rsrv_for_rep,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
).merge(
    aspa_def_df_for_rep,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
)

accoutning_7_for_rep_rec_cat['aspatially_deferred_aflb_ha'].fillna(0, inplace=True)
accoutning_7_for_rep_rec_cat['secondary_reserve_aflb_ha'].fillna(0, inplace=True)
accoutning_7_for_rep_rec_cat['accum_aflb_ha']=accoutning_7_for_rep_rec_cat['aspatially_deferred_aflb_ha']+accoutning_7_for_rep_rec_cat['secondary_reserve_aflb_ha'].fillna(0)
accoutning_7_for_rep_rec_cat['Percent of target']=accoutning_7_for_rep_rec_cat['accum_aflb_ha']/accoutning_7_for_rep_rec_cat['aflb_target_ha']*100


#### 8.	How much of each layer below is covered by Conservation Lands [note update in method in 3.o where only aspatially deferred old in TSUs contributes to Conservation Lands- marxan inputs

a.	Each WMB’s AFLB<br>
b.	Each TSU’s AFLB<br>
c.	Each Rec_Cat/FOR REP category for each WMB<br>
d.	CIA1/CIA2/CIAP<br>
e.	Each of the Optimal Lands inputs<br>


In [ ]:
conservation_lands_8a_wmb=consrv_lnd_df.groupby(
    ['NAME'], as_index=False).agg(
        Conservation_Lands_Total_Area_ha=('area_ha', 'sum'),
        Conservation_Lands_AFLB_Area_ha=('aflb_ha', 'sum'),
        Conservation_Lands_THLB_Area_ha=('thlb_area_ha','sum')
        )

accounting_8a_wmb=wmb_df.merge(
    conservation_lands_8a_wmb,
    on=['NAME'],
    how='left'
)

conservation_lands_8b_tsu=consrv_lnd_df.groupby(
    ['NAME', 'TRAPLINE_AREA_IDENTIFIER'], as_index=False).agg(
        Conservation_Lands_Total_Area_ha=('area_ha', 'sum'),
        Conservation_Lands_AFLB_Area_ha=('aflb_ha', 'sum'),
        Conservation_Lands_THLB_Area_ha=('thlb_area_ha','sum')
        )
accounting_8b_tsu=tsu_df.merge(
    conservation_lands_8b_tsu,
    on=['NAME', 'TRAPLINE_AREA_IDENTIFIER'],
    how='left'
)


In [ ]:
#8d/e
if "cia" not in globals():
    cia=gpd.read_parquet(os.path.join(wmv_inputs,'cia_1_p.parquet'))
    info('cia 1 p')
if "licks" not in globals():
    licks=gpd.read_parquet(os.path.join(wmv_inputs,'mineral_licks.parquet'))
    info('licks')
if "cia2" not in globals():
    cia2 =gpd.read_parquet(os.path.join(wmv_inputs,'cia_2.parquet'))
    info('cia 2')
if "low_cost" not in globals():
    low_cost = gpd.read_parquet(os.path.join(output_dir, "step_3e_Low_Cost_Lands.parquet"))
    low_cost=low_cost.dissolve()
    info('low cost')
if "headwater_gdf" not in globals():
    headwater_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'headwaters.parquet'))
    info('headwaters')
if "riparian_gdf" not in globals():
    riparian_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'riparian.parquet'))
    info('rip')
if "wetlands_gdf" not in globals():
    wetlands_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'wetlands.parquet'))
    info('wetlands')
if "moose_gdf" not in globals():
    moose_gdf=gpd.read_parquet(os.path.join(wmv_inputs,'moose.parquet'))
    info('moose')
if "caribou1" not in globals():
    caribou1=gpd.read_parquet(os.path.join(wmv_inputs,'caribou_1.parquet'))
    info('caribou 1')
if "caribou2" not in globals():
    caribou2=gpd.read_parquet(os.path.join(wmv_inputs,'caribou_2.parquet'))
    info('caribou 2')
if "fisher" not in globals():
    fisher=gpd.read_parquet(os.path.join(wmv_inputs,'fisher.parquet'))

In [ ]:
def fix(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf is None or gdf.empty: 
        return gdf
    try:
        gdf["geometry"] = gdf.geometry.apply(lambda x: make_valid(x) if x and not x.is_valid else x)
    except Exception:
        gdf["geometry"] = gdf.buffer(0)
    return gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

def to3005(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf is None or gdf.empty: 
        return gdf
    return gdf if (gdf.crs and str(gdf.crs).endswith("3005")) else gdf.to_crs("EPSG:3005")

def geom_category(gdf: gpd.GeoDataFrame) -> str:
    """Return 'polygon' | 'line' | 'point' based on observed geometry types."""
    if gdf is None or gdf.empty: 
        return "unknown"
    gt = set(getattr(g, "geom_type", "").lower() for g in gdf.geometry.values if g is not None)
    if any("polygon" in t for t in gt): return "polygon"
    if any("line" in t for t in gt):    return "line"
    if any("point" in t for t in gt):   return "point"
    return "unknown"

def dissolve_union(gdf: gpd.GeoDataFrame):
    """Return a single (multi)geometry union for fast vectorized intersection, or None if empty."""
    if gdf is None or gdf.empty: 
        return None
    geom_list = [g for g in gdf.geometry.values if g is not None and not g.is_empty]
    if not geom_list:
        return None
    u = shapely.union_all(geom_list)
    return u if (u is not None and not shapely.is_empty(u)) else None

def coverage_stats(feature_name: str,
                   feat_gdf: gpd.GeoDataFrame,
                   cons_union_geom,
                   clip_to_wmb: gpd.GeoDataFrame | None = None) -> dict:
    """
    Compute percent coverage of feature by Conservation Lands, handling polygons, lines, points.
    Returns a dict row suitable for tabular summary.
    """
    row = {"feature": feature_name,
           "metric": None,
           "total": np.nan,
           "in_conservation": np.nan,
           "percent": np.nan}

    if feat_gdf is None or feat_gdf.empty or cons_union_geom is None or shapely.is_empty(cons_union_geom):
        return row

    # Fix/CRS
    feat = to3005(fix(feat_gdf))

    # Optional clipping to WMB if provided
    if clip_to_wmb is not None and not clip_to_wmb.empty:
        feat = gpd.overlay(feat[["geometry"]], clip_to_wmb[["geometry"]], how="intersection", keep_geom_type=True)
        if feat.empty:
            return row

    kind = geom_category(feat)

    if kind == "polygon":
        inter_geoms = shapely.intersection(feat.geometry.values, cons_union_geom)
        total_area = shapely.area(feat.geometry.values).sum()       # m²
        in_area    = shapely.area(inter_geoms).sum()                 # m²
        total_ha   = total_area / 10_000.0
        in_ha      = in_area / 10_000.0
        pct        = (in_ha / total_ha) if total_ha > 0 else np.nan
        row.update({"metric": "area_ha", "total": total_ha, "in_conservation": in_ha, "percent": pct})

    elif kind == "line":
        inter_geoms = shapely.intersection(feat.geometry.values, cons_union_geom)
        total_len = shapely.length(feat.geometry.values).sum()      # m
        in_len    = shapely.length(inter_geoms).sum()                # m
        total_km  = total_len / 1000.0
        in_km     = in_len / 1000.0
        pct       = (in_km / total_km) if total_km > 0 else np.nan
        row.update({"metric": "length_km", "total": total_km, "in_conservation": in_km, "percent": pct})

    elif kind == "point":
        # Use covers so boundary points are counted as inside
        mask_inside = shapely.covers(cons_union_geom, feat.geometry.values)
        total_n = len(feat)
        in_n    = int(np.count_nonzero(mask_inside))
        pct     = (in_n / total_n) if total_n > 0 else np.nan
        row.update({"metric": "count", "total": total_n, "in_conservation": in_n, "percent": pct})

    # else: unknown -> keep NaNs

    return row

In [ ]:
clip_wmb = None  

consl = to3005(fix(consrv_lnd_df))
cons_union_geom = dissolve_union(consl)
if cons_union_geom is None:
    raise ValueError("[3l] Conservation Lands union is empty; cannot compute coverage.")

features = [
    ("CIA1",        cia),
    ("Mineral Licks",            licks),
    ("CIA2",       cia2),
    ("Low-Cost Lands",           low_cost),        # already dissolved, good
    ("Headwaters",               headwater_gdf),
    ("Riparian",                 riparian_gdf),
    ("Wetlands",                 wetlands_gdf),
    ("Moose Habitat",            moose_gdf),
    ("Caribou (1)",              caribou1),
    ("Caribou (2)",              caribou2),
    ("Fisher Habitat",           fisher),
]

rows = []
for name, gdf in features:
    try:
        row = coverage_stats(name, gdf, cons_union_geom, clip_to_wmb=clip_wmb)
        rows.append(row)
        pct = row["percent"]
        pct_str = f"{pct:.3%}" if pd.notna(pct) else "NA"
        total_str = f"{row['total']:.3f}" if pd.notna(row["total"]) else "NA"
        in_str    = f"{row['in_conservation']:.3f}" if pd.notna(row["in_conservation"]) else "NA"
        info(f"{name}: metric={row['metric']} | total={total_str} | in_cons={in_str} | pct={pct_str}")
    except Exception as e:
        info(f"{name}: FAILED ({e})")

summary_df = pd.DataFrame(rows)

summary_df["percent_pct"] = (summary_df["percent"] * 100.0).round(2)
summary_df["total_round"] = summary_df["total"].round(3)



accounting_8d_e_optimal=summary_df
accounting_8d_e_optimal.to_csv(os.path.join(table_outputs,'accounting_8d_e_optimal.csv'))

In [ ]:
ret_class_gdf=gpd.read_parquet(ret_class)

In [ ]:
if 'NAME' not in ret_class_gdf.columns:
    ret_class_gdf.rename(columns={'WATER_MANAGEMENT_BASIN_NAME':'NAME'}, inplace=True)

ret_class_gdf['aflb_area_ha']=ret_class_gdf['aflb_fact']*(ret_class_gdf.geometry.area/10000)
ret_class_gdf['area_ha']=(ret_class_gdf.geometry.area/10000)

if 'NAME' not in ret_class_gdf.columns:
    ret_class_gdf.rename(columns={'WATER_MANAGEMENT_BASIN_NAME':'NAME'}, inplace=True)

ret_class_gdf_g=ret_class_gdf.groupby(['NAME','Rec_Cat','Rec_Cat_short'], as_index=False).agg(
        recruitment_available_area_ha=('aflb_area_ha','sum'),
        recruitment_available_aflb_area_ha=('aflb_area_ha','sum')
    )



In [ ]:
aspa_def_joiend= aspa_def_df.sjoin(consrv_lnd_df, how="left", predicate="intersects")
aspa_def_missing = aspa_def_joiend[aspa_def_joiend["index_right"].isna()]

aspa_def_missing.rename(columns={'NAME_left':'NAME','Rec_Cat_left':'Rec_Cat','Rec_Cat_short_left':'Rec_Cat_short',
                                 'area_ha_left':'area_ha','aflb_ha_left':'aflb_ha','thlb_area_ha_left':'thlb_area_ha'}, inplace=True)

aspa_def_missing_for_rep=aspa_def_missing.groupby(['NAME','Rec_Cat','Rec_Cat_short'], as_index=False).agg(
    aspatially_deferred_area_outside_con=('area_ha','sum'),
    aspatially_deferred_aflb_ha_outside_con=('aflb_ha', 'sum'),
    aspatially_deferred_thlb_ha_outside_con=('thlb_area_ha','sum')
)


In [ ]:

#8c 
consrv_lnd_df_for_rep=consrv_lnd_df.groupby(['NAME','Rec_Cat','Rec_Cat_short'], as_index=False).agg(
    conservation_land_area=('area_ha','sum'),
    conservation_land_aflb_ha=('aflb_ha', 'sum'),
    conservation_land_thlb_ha=('thlb_area_ha','sum')
)


if 'NAME' not in wmb_for_rep_df.columns:
    wmb_for_rep_df.rename(columns={'WATER_MANAGEMENT_BASIN_NAME':'NAME'}, inplace=True)

accoutning_8c_for_rep_rec_cat=wmb_for_rep_df.merge(
    ret_class_gdf_g,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
).merge(
    consrv_lnd_df_for_rep,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
).merge(
    aspa_def_missing_for_rep,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
)
accoutning_8c_for_rep_rec_cat=accoutning_8c_for_rep_rec_cat.fillna(0)

accoutning_8c_for_rep_rec_cat['conservation_AD_total_area']=accoutning_8c_for_rep_rec_cat['conservation_land_area']+accoutning_8c_for_rep_rec_cat['aspatially_deferred_area_outside_con']
accoutning_8c_for_rep_rec_cat['conservation_AD_total_aflb_area']=accoutning_8c_for_rep_rec_cat['conservation_land_aflb_ha']+accoutning_8c_for_rep_rec_cat['aspatially_deferred_aflb_ha_outside_con']
accoutning_8c_for_rep_rec_cat['conservation_lands_aflb_meet_target']=accoutning_8c_for_rep_rec_cat['conservation_land_aflb_ha']>=accoutning_8c_for_rep_rec_cat['aflb_target_ha']
accoutning_8c_for_rep_rec_cat['conservation_AD_meet_target']=accoutning_8c_for_rep_rec_cat['conservation_AD_total_aflb_area']>=accoutning_8c_for_rep_rec_cat['aflb_target_ha']



#### 11.	How much of each TSU and WMB AFLB is covered by ‘conservation lands’ and ‘aspatially deferred old forest’. What is the THLB impact outside HV1s? 

In [ ]:
if 'NAME_1' in consrv_lnd_df.columns:
    consrv_lnd_df.rename(columns={'NAME_1':'NAME'}, inplace=True)  

consrv_lnd_df_wmb=consrv_lnd_df.groupby(['NAME'], as_index=False).agg(
    conservation_land_area=('area_ha','sum'),
    conservation_land_aflb_ha=('aflb_ha', 'sum'),
    conservation_land_thlb_ha=('thlb_area_ha','sum')
)

aspa_def_missing_wmb=aspa_def_missing.groupby(['NAME'], as_index=False).agg(
    aspatially_deferred_area_outside_con=('area_ha','sum'),
    aspatially_deferred_aflb_ha_outside_con=('aflb_ha', 'sum'),
    aspatially_deferred_thlb_ha_outside_con=('thlb_area_ha','sum')
)

accoutning_11_wmb=wmb_df.merge(
    consrv_lnd_df_wmb,
    on=['NAME'],
    how='left'
).merge(
    aspa_def_missing_wmb,
    on=['NAME'],
    how='left'
)
accoutning_11_wmb=accoutning_11_wmb.fillna(0)


accoutning_11_wmb['conserv_difference_ha']=accoutning_11_wmb['aflb_25_percent']-accoutning_11_wmb['conservation_land_aflb_ha']
accoutning_11_wmb['conserv_coverage_pct_of_aflb']=accoutning_11_wmb['conservation_land_aflb_ha']/accoutning_11_wmb['SUM_aflb_ha']*100  
accoutning_11_wmb['conserv_meets_25_percent_aflb']=accoutning_11_wmb['conserv_coverage_pct_of_aflb']>=25 


accoutning_11_wmb['combined_total_area']=accoutning_11_wmb['conservation_land_area']+accoutning_11_wmb['aspatially_deferred_area_outside_con']
accoutning_11_wmb['combined_total_aflb_area']=accoutning_11_wmb['conservation_land_aflb_ha']+accoutning_11_wmb['aspatially_deferred_aflb_ha_outside_con']
accoutning_11_wmb['combined_difference_ha']=accoutning_11_wmb['aflb_25_percent']-accoutning_11_wmb['combined_total_aflb_area']
accoutning_11_wmb['combined_coverage_pct_of_aflb']=accoutning_11_wmb['combined_total_aflb_area']/accoutning_11_wmb['SUM_aflb_ha']*100  
accoutning_11_wmb['combined_meets_25_percent_aflb']=accoutning_11_wmb['combined_coverage_pct_of_aflb']>=25 

 

In [ ]:
if 'NAME_1' in consrv_lnd_df.columns:
    consrv_lnd_df.rename(columns={'NAME_1':'NAME'}, inplace=True)  

if 'TRAPLINE_AREA_IDENTIFIER_right' in aspa_def_missing.columns:
    aspa_def_missing.rename(columns={'TRAPLINE_AREA_IDENTIFIER_right':'TRAPLINE_AREA_IDENTIFIER'}, inplace=True)  

consrv_lnd_df_tsu=consrv_lnd_df.groupby(['NAME','TRAPLINE_AREA_IDENTIFIER'], as_index=False).agg(
    conservation_land_area=('area_ha','sum'),
    conservation_land_aflb_ha=('aflb_ha', 'sum'),
    conservation_land_thlb_ha=('thlb_area_ha','sum')
)

aspa_def_missing_tsu=aspa_def_missing.groupby(['NAME','TRAPLINE_AREA_IDENTIFIER'], as_index=False).agg(
    aspatially_deferred_area_outside_con=('area_ha','sum'),
    aspatially_deferred_aflb_ha_outside_con=('aflb_ha', 'sum'),
    aspatially_deferred_thlb_ha_outside_con=('thlb_area_ha','sum')
)

accoutning_11_tsu=tsu_df.merge(
    consrv_lnd_df_tsu,
    on=['NAME','TRAPLINE_AREA_IDENTIFIER'],
    how='left'
).merge(
    aspa_def_missing_tsu,
    on=['NAME','TRAPLINE_AREA_IDENTIFIER'],
    how='left'
)
accoutning_11_tsu=accoutning_11_tsu.fillna(0)


accoutning_11_tsu['conserv_difference_ha']=accoutning_11_tsu['tsu_33_aaflb_ha']-accoutning_11_tsu['conservation_land_aflb_ha']
accoutning_11_tsu['conserv_coverage_pct_of_aflb']=accoutning_11_tsu['conservation_land_aflb_ha']/accoutning_11_tsu['tsu_aflb_ha']*100  
accoutning_11_tsu['conserv_meets_25_percent_aflb']=accoutning_11_tsu['conserv_coverage_pct_of_aflb']>=25 


accoutning_11_tsu['combined_total_area']=accoutning_11_tsu['conservation_land_area']+accoutning_11_tsu['aspatially_deferred_area_outside_con']
accoutning_11_tsu['combined_total_aflb_area']=accoutning_11_tsu['conservation_land_aflb_ha']+accoutning_11_tsu['aspatially_deferred_aflb_ha_outside_con']
accoutning_11_tsu['combined_difference_ha']=accoutning_11_tsu['tsu_33_aaflb_ha']-accoutning_11_tsu['combined_total_aflb_area']
accoutning_11_tsu['combined_coverage_pct_of_aflb']=accoutning_11_tsu['combined_total_aflb_area']/accoutning_11_tsu['tsu_aflb_ha']*100  
accoutning_11_tsu['combined_meets_25_percent_aflb']=accoutning_11_tsu['combined_coverage_pct_of_aflb']>=25 

In [ ]:
#11 THLB
#totals inclusive 
consrv_lnd_area=consrv_lnd_df['area_ha'].sum()
consrv_lnd_aflb_area=consrv_lnd_df['aflb_ha'].sum()
consrv_lnd_area_thlb_area=consrv_lnd_df['thlb_area_ha'].sum()

aspa_def_missing_area=aspa_def_missing['area_ha'].sum()
aspa_def_missing_aflb_area=aspa_def_missing['aflb_ha'].sum()
aspa_def_missing_area_thlb_area=aspa_def_missing['thlb_area_ha'].sum()
if 'TRAPLINE_AREA_IDENTIFIER_left' in aspa_def_missing.columns:
    aspa_def_missing.rename(columns={'TRAPLINE_AREA_IDENTIFIER_left':'TRAPLINE_AREA_IDENTIFIER'}, inplace=True)


combined_area=consrv_lnd_area+aspa_def_missing_area
combined_aflb_area=consrv_lnd_aflb_area+aspa_def_missing_aflb_area
combined_thlb_area=consrv_lnd_area_thlb_area+aspa_def_missing_area_thlb_area

#11 THLB
#totals without hv1
aspa_def_missing=aspa_def_missing[['area_ha','thlb_area_ha', 'aflb_ha', 'TRAPLINE_AREA_IDENTIFIER','NAME','Rec_Cat', 'Rec_Cat_short','geometry']].copy()

consrv_lnd_joined = consrv_lnd_df.sjoin(hv1, how="left", predicate="intersects")
consrv_lnd_missing = consrv_lnd_joined[consrv_lnd_joined["index_right"].isna()]

aspa_def_missing_joined = aspa_def_missing.sjoin(hv1, how="left", predicate="intersects")
aspa_def_missing_lnd_missing = aspa_def_missing_joined[aspa_def_missing_joined["index_right"].isna()]

consrv_lnd_area_no_hv1=consrv_lnd_missing['area_ha_left'].sum()
consrv_lnd_aflb_area_no_hv1=consrv_lnd_missing['aflb_ha'].sum()
consrv_lnd_area_thlb_area_no_hv1=consrv_lnd_missing['thlb_area_ha'].sum()

aspa_def_missing_area_no_hv1=aspa_def_missing_lnd_missing['area_ha_left'].sum()
aspa_def_missing_aflb_area_no_hv1=aspa_def_missing_lnd_missing['aflb_ha'].sum()
aspa_def_missing_area_thlb_area_no_hv1=aspa_def_missing_lnd_missing['thlb_area_ha'].sum()

combined_area_no_hv1=consrv_lnd_area_no_hv1+aspa_def_missing_area_no_hv1
combined_aflb_area_no_hv1=consrv_lnd_aflb_area_no_hv1+aspa_def_missing_aflb_area_no_hv1
combined_thlb_area_no_hv1=consrv_lnd_area_thlb_area_no_hv1+aspa_def_missing_area_thlb_area_no_hv1

accoutning_11_thlb = pd.DataFrame({
    "Source": [
        "Conservation Land",
        "ASPA Deferral outside conservg",
        "Combined",
    ],
    "Inclusive - Total Area (ha)": [
        consrv_lnd_area,
        aspa_def_missing_area,
        combined_area,
    ],
    "Inclusive - AFLB Area (ha)": [
        consrv_lnd_aflb_area,
        aspa_def_missing_aflb_area,
        combined_aflb_area,
    ],
    "Inclusive - THLB Area (ha)": [
        consrv_lnd_area_thlb_area,
        aspa_def_missing_area_thlb_area,
        combined_thlb_area,
    ],
    "No HV1 - Total Area (ha)": [
        consrv_lnd_area_no_hv1,
        aspa_def_missing_area_no_hv1,
        combined_area_no_hv1,
    ],
    "No HV1 - AFLB Area (ha)": [
        consrv_lnd_aflb_area_no_hv1,
        aspa_def_missing_aflb_area_no_hv1,
        combined_aflb_area_no_hv1,
    ],
    "No HV1 - THLB Area (ha)": [
        consrv_lnd_area_thlb_area_no_hv1,
        aspa_def_missing_area_thlb_area_no_hv1,
        combined_thlb_area_no_hv1,
    ],
})



In [ ]:
accounting_8d_e_optimal=pd.read_csv(os.path.join(table_outputs,'accounting_8d_e_optimal.csv'))

In [ ]:

accounting_name_map = {
    "3_tsu":accounting_3_tsu,
    "3_wmb":accounting_3_wmb,
    "4_wmb_recruitment":accounting_4_wmb_recruitment,
    "5_tsu":accounting_5_tsu,
    "5_wmb":accounting_5_wmb,
    "6_thlb":account_6_thlb,
    "7_for_rep_rec_cat":accoutning_7_for_rep_rec_cat,
    "8a_wmb":accounting_8a_wmb,
    "8b_tsu":accounting_8b_tsu,
    "8d_e_optimal":accounting_8d_e_optimal,
    "8c_for_rep_rec_cat":accoutning_8c_for_rep_rec_cat,
    "11_tsu":accoutning_11_tsu,
    "11_wmb":accoutning_11_wmb,
    "11_thlb":accoutning_11_thlb
}

datenow=datetime.today().strftime('%Y%m%d')
excel_out=os.path.join(table_outputs,f"accounting_{scen5_run_number}_{datenow}.xlsx")

with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    for sheet_name, df in accounting_name_map.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
